# FLASK RQ1–RQ3 Colab runner

This notebook runs the GPU-heavy FLASK extraction jobs and packages artifacts back to Google Drive.

Security note: complete Google / Drive / Hugging Face authentication inside this Colab notebook or your browser. Do not paste verification codes, cookies, or account passwords into chat.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/oceanusXXD/Dual-Space-Prototypical-Probing-for-OOD-Robust-LLM-Evaluation'
REPO_DIR = Path('/content/Dual-Space-Prototypical-Probing-for-OOD-Robust-LLM-Evaluation')
DRIVE_ROOT = Path('/content/drive/MyDrive/llm_judge_ood_colab')
INPUT_PACKAGES = DRIVE_ROOT / 'input_packages'
OUTPUT_PACKAGES = DRIVE_ROOT / 'output_packages'
MODEL_ROOT = DRIVE_ROOT / 'models'

# Toggle only what you still need. Jobs are resumable when their output dirs already contain manifests/parts.
RUN_QWEN08B_FULL_STRICT_PRELOGIT_BSPACE = False  # already completed locally in current repo snapshot
RUN_QWEN4B_FULL_A_SPACE = False                 # already completed locally in current repo snapshot
RUN_QWEN4B_FULL_B_SPACE_MASKED_MEAN = True      # GPU-heavy 4B full B-space extractor
RUN_CPU_FULL_VIM_AFTER_EXTRACTION = True

QWEN08B_REVISION = '2fc06364715b967f1860aea9cf38778875588b17'
QWEN4B_REVISION = '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a'

INPUT_PACKAGES.mkdir(parents=True, exist_ok=True)
OUTPUT_PACKAGES.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

In [ ]:
import os, subprocess, textwrap

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

os.chdir(REPO_DIR)
print('Repo:', REPO_DIR)
subprocess.run(['git', 'status', '--short'], check=False)

In [ ]:
import subprocess, sys

subprocess.run(['apt-get', 'update'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'zstd', 'git-lfs'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'huggingface_hub', 'accelerate', 'transformers', 'safetensors', 'joblib', 'scikit-learn', 'zstandard'], check=True)

## Restore input artifacts

Upload any existing `.tar.zst` packages to `MyDrive/llm_judge_ood_colab/input_packages/`. The notebook extracts them into the repo root. This is optional if the repo already has the needed datasets/artifacts.

In [ ]:
import subprocess
from pathlib import Path

archives = sorted(INPUT_PACKAGES.glob('*.tar.zst'))
print('archives:', [str(p) for p in archives])
for archive in archives:
    print('extracting', archive)
    subprocess.run(['tar', '--zstd', '-xf', str(archive), '-C', str(REPO_DIR)], check=True)

## Download model snapshots

If Hugging Face asks for authentication, run `huggingface-cli login` in this notebook and enter the token in Colab only. The repository scripts use `local_files_only=True`, so the snapshots must exist at the paths below before extraction.

In [ ]:
from huggingface_hub import snapshot_download

def ensure_snapshot(repo_id, revision, local_dir):
    local_dir = Path(local_dir)
    if (local_dir / 'config.json').exists():
        print('model exists:', local_dir)
        return local_dir
    print('downloading', repo_id, revision, '->', local_dir)
    snapshot_download(repo_id=repo_id, revision=revision, local_dir=str(local_dir), local_dir_use_symlinks=False)
    return local_dir

QWEN08B_DIR = MODEL_ROOT / 'qwen3.5-0.8b'
QWEN4B_DIR = MODEL_ROOT / 'qwen3.5-4b'

if RUN_QWEN08B_FULL_STRICT_PRELOGIT_BSPACE:
    ensure_snapshot('Qwen/Qwen3.5-0.8B', QWEN08B_REVISION, QWEN08B_DIR)
if RUN_QWEN4B_FULL_A_SPACE or RUN_QWEN4B_FULL_B_SPACE_MASKED_MEAN:
    ensure_snapshot('Qwen/Qwen3.5-4B', QWEN4B_REVISION, QWEN4B_DIR)

## GPU extraction jobs

In [ ]:
import subprocess, sys

def run(cmd):
    print('\n$ ' + ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

if RUN_QWEN08B_FULL_STRICT_PRELOGIT_BSPACE:
    run([
        sys.executable, 'scripts/llm_judge_ood/62_run_flask_qwen35_08b_digit_strict_prelogit_bspace.py',
        '--scope', '10x12',
        '--model-path', QWEN08B_DIR,
        '--attn-implementation', 'sdpa',
        '--batch-size', '1024',
        '--max-batch-tokens', '262144',
    ])

if RUN_QWEN4B_FULL_A_SPACE:
    run([
        sys.executable, 'scripts/llm_judge_ood/57_extract_flask_full_a_space.py',
        '--model-path', QWEN4B_DIR,
        '--attn-implementation', 'sdpa',
        '--batch-size', '512',
        '--max-batch-tokens', '131072',
    ])

if RUN_QWEN4B_FULL_B_SPACE_MASKED_MEAN:
    run([
        sys.executable, 'scripts/llm_judge_ood/52_extract_flask_full_b_space.py',
        '--model-path', QWEN4B_DIR,
        '--attn-implementation', 'sdpa',
        '--batch-size', '512',
        '--max-batch-tokens', '131072',
    ])

## CPU post-processing / detector smoke runs

In [ ]:
if RUN_CPU_FULL_VIM_AFTER_EXTRACTION:
    # 0.8B strict-prelogit full 10x12, if available.
    p08 = Path('artifacts/flask_full_validation/qwen35_08b_10x12_digit_direct_judge_strict_prelogit_bspace/strict_final_prelogit_b_space_features.npz')
    if p08.exists():
        run([
            sys.executable, 'scripts/llm_judge_ood/63_run_flask_basic_vim.py',
            '--features', p08,
            '--output-dir', 'artifacts/flask_full_validation/basic_vim_08b_10x12_strict_prelogit',
            '--layer', 'last',
            '--overwrite',
        ])

    # 4B full B-space from script 52, if available.
    p4b = Path('artifacts/flask_full_b_space/unique_b_space_hidden_states.npz')
    if p4b.exists():
        run([
            sys.executable, 'scripts/llm_judge_ood/63_run_flask_basic_vim.py',
            '--features', p4b,
            '--output-dir', 'artifacts/flask_full_validation/basic_vim_4b_10x12_masked_mean',
            '--layer', 'last',
            '--overwrite',
        ])

## Package outputs to Drive

In [ ]:
import datetime, subprocess

stamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
out = OUTPUT_PACKAGES / f'flask_rq1_3_colab_outputs_{stamp}.tar.zst'
paths = [
    'artifacts/flask_full_b_space',
    'artifacts/flask_full_validation/basic_vim_08b_10x12_strict_prelogit',
    'artifacts/flask_full_validation/basic_vim_4b_10x12_masked_mean',
    'artifacts/flask_full_validation/qwen35_08b_10x12_digit_direct_judge_strict_prelogit_bspace',
    'artifacts/flask_full_a_space',
]
existing = [p for p in paths if Path(p).exists()]
print('packaging:', existing)
if existing:
    subprocess.run(['tar', '--zstd', '-cf', str(out), *existing], check=True)
    subprocess.run(['sha256sum', str(out)], check=True)
    print('wrote', out)
else:
    print('no output paths found')